# GroundTruth Validation Walkthrough

This notebook demonstrates the core GroundTruth pipeline:
1. Load a SaaSquatch CSV export
2. Match companies against PPP loan records
3. Compute revenue signals
4. Produce validation charts

In [ ]:
import sys
sys.path.insert(0, '..')

import csv
import math
from pathlib import Path

from groundtruth.csv_parser import parse_csv, parse_revenue_string
from groundtruth.match import match_business
from groundtruth.signals import compute_signals

## 1. Load Sample Export

In [ ]:
sample_path = Path('../data/samples/saasquatch_export_sample.csv')
content = sample_path.read_text()
rows, mapping = parse_csv(content)

print(f'Loaded {len(rows)} rows')
print(f'Column mapping:')
for field, m in mapping.items():
    print(f'  {field}: {m.source_column} ({m.confidence})')

## 2. Match Against PPP Records

In [ ]:
results = []
for i, row in enumerate(rows):
    bm = match_business(
        name=row.company_name,
        city=row.city,
        state=row.state,
        zip_code=row.zip_code,
        address=row.address,
        industry=row.industry,
    )
    signals = compute_signals(
        match=bm.best_match,
        sba_loans=bm.sba_loans,
    )
    sq_rev = parse_revenue_string(row.estimated_revenue)
    results.append({
        'name': row.company_name,
        'city': row.city,
        'state': row.state,
        'match_tier': bm.best_match.tier if bm.best_match else 'No match',
        'match_confidence': bm.best_match.confidence if bm.best_match else 0,
        'ppp_revenue': signals.revenue.revenue_point,
        'sq_revenue': sq_rev,
        'headcount': signals.headcount,
        'recommendation': signals.recommendation,
    })
    if (i + 1) % 20 == 0:
        print(f'Processed {i+1}/{len(rows)}')

matched = [r for r in results if r['match_tier'] != 'No match']
print(f'\nMatched: {len(matched)} / {len(results)}')

## 3. Results Summary

In [ ]:
from collections import Counter

tiers = Counter(r['match_tier'] for r in results)
recs = Counter(r['recommendation'] for r in results)

print('Match Tiers:')
for tier, count in tiers.most_common():
    print(f'  {tier}: {count}')

print('\nRecommendations:')
for rec, count in recs.most_common():
    print(f'  {rec}: {count}')

print('\nTop Matches:')
for r in sorted(matched, key=lambda x: x['match_confidence'], reverse=True)[:10]:
    rev_str = f"${r['ppp_revenue']/1e6:.1f}M" if r['ppp_revenue'] else '—'
    print(f"  {r['name']:40s} {r['match_tier']:10s} {r['match_confidence']:.0%}  Rev: {rev_str}")

## 4. Validation Charts

Compare SaaSquatch revenue estimates against PPP-implied revenue for matched leads.

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import numpy as np
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print('matplotlib not installed — charts will be text-based')

# Filter to rows with both revenue values
valid = [r for r in matched
         if r['ppp_revenue'] and r['ppp_revenue'] > 0
         and r['sq_revenue'] and r['sq_revenue'] > 0
         and r['match_tier'] in ('High', 'Probable')]

print(f'{len(valid)} leads with both revenue estimates and High/Probable match')

if valid and HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Scatter: PPP vs SaaSquatch
    ppp_revs = [r['ppp_revenue'] for r in valid]
    sq_revs = [r['sq_revenue'] for r in valid]

    ax1.scatter(ppp_revs, sq_revs, alpha=0.7, c='#2563eb', s=40)
    mn = min(min(ppp_revs), min(sq_revs))
    mx = max(max(ppp_revs), max(sq_revs))
    ax1.plot([mn, mx], [mn, mx], 'k--', alpha=0.3, label='Perfect agreement')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('PPP-Implied Revenue')
    ax1.set_ylabel('SaaSquatch Estimate')
    ax1.set_title('Revenue Comparison (log-log)')
    ax1.legend()

    # Histogram: log ratio
    log_ratios = [math.log(sq / ppp) for sq, ppp in zip(sq_revs, ppp_revs)]
    ax2.hist(log_ratios, bins=10, color='#2563eb', alpha=0.8, edgecolor='white')
    ax2.axvline(x=math.log(2), color='red', linestyle='--', alpha=0.5, label='>2x over')
    ax2.axvline(x=-math.log(2), color='red', linestyle='--', alpha=0.5, label='>2x under')
    ax2.set_xlabel('ln(SaaSquatch / PPP)')
    ax2.set_ylabel('Count')
    ax2.set_title('Estimate Disagreement')
    ax2.legend()

    plt.tight_layout()
    plt.savefig('validation_charts.png', dpi=150)
    plt.show()
    print('Charts saved to validation_charts.png')

## 5. Headline Stat

In [ ]:
if valid:
    off_by_2x = sum(1 for r in valid
                    if abs(math.log(r['sq_revenue'] / r['ppp_revenue'])) > math.log(2))
    pct = off_by_2x / len(valid) * 100
    print(f'On {len(valid)} matched leads, the platform\'s revenue estimate '
          f'disagreed with the PPP-implied figure by more than 2x '
          f'for {off_by_2x} leads ({pct:.0f}%).')
    if off_by_2x > 0:
        print(f'Those {off_by_2x} leads represent {off_by_2x} enrichment credits '
              f'that may need verification before spending.')
else:
    print('No valid revenue pairs found in demo subset.')
    print('With the full PPP dataset (11.5M rows), matches would be much higher.')
    print('Run: python scripts/ingest.py --states CA,TX,FL,AZ')
    print('Then set GROUNDTRUTH_DATA_FULL=1 to use the full dataset.')